In [1]:
### follow this notebook fo rmultiple charge trap implementation

In [2]:
import time
import sys
import os

# Add the parent directory to sys.path
notebook_path = os.getcwd()  # Get the current working directory where the notebook is located
parent_directory = os.path.dirname(notebook_path)  # Move to the parent directory
sys.path.append(parent_directory)

# Now you can import config
from config import *
#from Minimal_Model_2D import *
from Minimal_Model_2D import *
import time

np.set_printoptions(threshold=sys.maxsize)

In [3]:
class Minimal_Model_multiple_charge_trap_potential(twoD_Minimal_Model):

    ### overriding a H same spin method to include multiple charge trap 

    def H_same_spin(self, x_ct= np.empty(5), y_ct=np.empty(5), z_ct=np.empty(5), z_im=np.empty(5), z_qw=0, dis_mag=0, uncorr_var=1, lmbda=0, seed=None, kx=0, ky=0):

        #print('a- same spin class overridden')

        M = (hbm/self.meff_sm* self.twodOP.KxSq(kx) + hbm/self.meff_sm* self.twodOP.KySq(ky) - self.chem_pot() + 1/4/hbm*self.meff_sm*self.omega**2*self.twodOP.ysq()  +self.dis_pot_matrix(dis_mag) +self.Disorder_full(uncorr_var, lmbda, seed) )

        for i in range(x_ct.shape[0]):

            #print(i)
            M = M + self.charge_pot_lattice(x_ct[i], y_ct[i], z_ct[i], z_im[i], z_qw)
 
        return M


In [4]:
NSM = 60            # number of sm x lattice points
M = 60              # should be atleast thrice confinement length
AX = 5              # lattice spacing in nm
AY = 5              # lattice spacing in nm
DELTA = 0.165       # in meV
MEFF_SC = 0.05    # proximitized Ge
MEFF_SM = 0.05     # Ge
NSC = 800
NTOT = 2*NSC + NSM
PHI = 0.5*np.pi 
ALPHAT = 400        # meV-nm^3 # 20 is within perturbative regime for widths atleast up to 100 nm
K_NUM = 8           # No of eigenvalues needed to be found 
#W_MIN = 50

LY = 100           # Width in nm
OMEGA = 2*4*hbm/LY**2/MEFF_SC

MY= 5 

print('lengths=', M*AY, LY)
OMEGA = 2*4*hbm/LY**2/MEFF_SC

print('subband energies=', OMEGA/2, 3*OMEGA/2)

print('4t=', 4*hbm/AX**2/MEFF_SM)

MU_SC =   OMEGA/2 +OMEGA/10  # in between two subbands - such that we get two abs prferably 
MU_SM =   OMEGA/2 +OMEGA/10  # in between two subbands - such that we get two abs prferably

print('mu = ', MU_SM)


UNCORR_VAR = 0  ### uncorrelated variance of disorder
LMBDA = 0
SEED = 58  ## seed for the disorder

# Define the range and the size of the array
x_low = (NSC+1)*AX  # Start of the range (inclusive)
x_high = (NSC+NSM+1)*AX  # End of the range (exclusive)

y_low = 0.0  # Start of the range (inclusive)
y_high = M*AY  # End of the range (exclusive)


z_low = 50.0  # Start of the range (inclusive)
z_high = 51.5  # End of the range (exclusive)

size = 5  # Size of the array

# Generate the array
X_CT = np.random.uniform(x_low, x_high, size)
Y_CT = np.random.uniform(y_low, y_high, size)
Z_CT = np.random.uniform(z_low, z_high, size)
Z_dis = np.ones(size)*(z_high+0.1) - Z_CT
Z_IM  = np.ones(size)*(z_high+0.1) + Z_dis

print(X_CT)
print(Y_CT)
print(Z_CT)
print(Z_IM)


ModCT = Minimal_Model_multiple_charge_trap_potential(NTOT, NSM, NSC, M, AX, AY, OMEGA, ALPHAT, DELTA, MEFF_SC, MEFF_SM, MU_SC, MU_SM)

lengths= 300 100
subband energies= 0.30505912614720065 0.915177378441602
4t= 122.02365045888027
mu =  0.3660709513766408
[4254.75309711 4012.84577201 4047.86537491 4079.59809288 4151.14633993]
[295.88541707  62.18303585  21.19401155  20.23422622  94.46227891]
[51.4958695  51.0948679  50.06960579 50.29521929 51.14159708]
[51.7041305  52.1051321  53.13039421 52.90478071 52.05840292]


In [ ]:

H = ModCT.H_sys(PHI, 'cu', X_CT, Y_CT, Z_CT, Z_IM)

E = eigsh(H, k=K_NUM, sigma=0, return_eigenvectors=False )

Es = np.sort(E)/DELTA

#Ep[i,:] = Es

print(Es)

#print(H.shape)

In [5]:
ITER = 21

K_NUM = 4

Ep = np.empty((ITER,K_NUM))

z_offset2 = 2*(z_high + 0.1)

for i in range(ITER):

    # Generate the array
    X_CT = np.random.uniform(x_low, x_high, size)
    Y_CT = np.random.uniform(y_low, y_high, size)
    Z_CT = np.random.uniform(z_low, z_high, size)
    Z_IM = z_offset2 -Z_CT  
        
    H = ModCT.H_sys(PHI, 'cu', X_CT, Y_CT, Z_CT, Z_IM)

    E = eigsh(H, k=K_NUM, sigma=0, return_eigenvectors=False )

    Es = np.sort(E)/DELTA

    Ep[i,:] = Es

    if i%5==0:

        print(Es)

    #print(H.shape)

    
np.save("cubic_spin_splitting_{}_charge_traps_{}_iter.npy".format(size,ITER),Ep)


[-0.13192261 -0.1317863   0.1317863   0.13192261]
[-0.16820595 -0.16808997  0.16808997  0.16820595]
[-0.17511356 -0.17498583  0.17498583  0.17511356]
[-0.09795129 -0.09773942  0.09773942  0.09795129]
[-0.24976691 -0.2496683   0.2496683   0.24976691]


In [6]:
Ep_splitting = Ep[:,K_NUM//2+1] - Ep[:, K_NUM//2]

In [7]:
Ep_splitting.shape

(21,)

In [8]:
np.mean(Ep_splitting)

0.00013196189082331712

In [9]:
np.var(Ep_splitting)

3.3086534749485614e-09

In [10]:
print(Ep_splitting)

[1.36309707e-04 2.90717085e-04 1.55967118e-04 9.59845071e-05
 7.01460117e-05 1.15973054e-04 1.51331706e-04 7.30570129e-05
 6.33491606e-05 6.08121549e-05 1.27737039e-04 1.08357095e-04
 1.03865710e-04 1.71493706e-04 2.36904192e-04 2.11869104e-04
 1.23576347e-04 1.05393603e-04 9.97578709e-05 1.69989911e-04
 9.86076136e-05]


In [11]:
print(Ep)

[[-0.13192261 -0.1317863   0.1317863   0.13192261]
 [-0.09159026 -0.09129954  0.09129954  0.09159026]
 [-0.14807051 -0.14791455  0.14791455  0.14807051]
 [-0.31509968 -0.3150037   0.3150037   0.31509968]
 [-0.23204089 -0.23197075  0.23197075  0.23204089]
 [-0.16820595 -0.16808997  0.16808997  0.16820595]
 [-0.18556864 -0.18541731  0.18541731  0.18556864]
 [-0.3538026  -0.35372954  0.35372954  0.3538026 ]
 [-0.40178512 -0.40172177  0.40172177  0.40178512]
 [-0.33471526 -0.33465444  0.33465444  0.33471526]
 [-0.17511356 -0.17498583  0.17498583  0.17511356]
 [-0.33155985 -0.3314515   0.3314515   0.33155985]
 [-0.22481756 -0.2247137   0.2247137   0.22481756]
 [-0.10197375 -0.10180226  0.10180226  0.10197375]
 [-0.09048275 -0.09024585  0.09024585  0.09048275]
 [-0.09795129 -0.09773942  0.09773942  0.09795129]
 [-0.23976663 -0.23964305  0.23964305  0.23976663]
 [-0.14867137 -0.14856598  0.14856598  0.14867137]
 [-0.28077385 -0.2806741   0.2806741   0.28077385]
 [-0.09238114 -0.09221115  0.09